In [ ]:
import pyodbc as odbc
import pandas as pd

conn = odbc.connect(
    """
    Driver={ODBC Driver 17 for SQL Server};
    Server=DESKTOP-H94GPH9;
    Database=ScuolaDb;
    Trusted_Connection=yes;
    TrustServerCertificate=yes;
    """
)

print("Connessione avvenuta con successo!")

Connessione avvenuta con successo!


RECUPERO DELLE TABELLE

SELECT TABLE_NAME: Definisce la colonna da recuperare. TABLE_NAME contiene il nome identificativo della tabella.

FROM INFORMATION_SCHEMA.TABLES: Indica la sorgente dei dati. TABLES è una vista di sistema che cataloga tutte le tabelle e le viste a cui l'utente ha accesso.

WHERE TABLE_TYPE = 'BASE TABLE': Questa clausola funge da filtro critico. Nel linguaggio SQL, una "Base Table" è una tabella che memorizza effettivamente i dati su disco, distinguendosi dalle "VIEW" (viste virtuali), che sono solo query salvate.

In [5]:
query = """
    SELECT TABLE_NAME
    FROM INFORMATION_SCHEMA.TABLES
    WHERE TABLE_TYPE = 'BASE TABLE'
    AND TABLE_CATALOG = 'ScuolaDb'
    AND TABLE_SCHEMA = 'dbo'
    ORDER BY TABLE_NAME
    """

df_tables = pd.read_sql(query, conn)

#lista delle tabelle

tabelle = df_tables["TABLE_NAME"].tolist()
print("Tabelle presenti nel database:")
for tabella in tabelle:
    print(tabella)

Tabelle presenti nel database:
Corso
Docente
Esame
Iscrizione
Studente
sysdiagrams


C:\Users\pietr\AppData\Local\Temp\ipykernel_16228\2283164734.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_tables = pd.read_sql(query, conn)


NOME FILE CON DATA

In [6]:
data_oggi = pd.to_datetime("today").strftime("%Y-%m-%d_%H-%M-%S")
print(f"Data di oggi: {data_oggi}")

file_excel = f"Backup_automatico_ScuolaDb3_{data_oggi}.xlsx"


Data di oggi: 2026-03-24_23-06-55


LISTA STATICA

In [7]:
stat =[]


EXPORT

In [9]:
with pd.ExcelWriter(file_excel, engine="openpyxl") as writer:
    for tabella in tabelle:
    
        query = f"SELECT * FROM {tabella}"
        df = pd.read_sql(query, conn)
        df.to_excel(writer, sheet_name=tabella, index=False)
        
        totale = len(df)        
        num_colonne = len(df.columns)
        
        stat.append(
            {
            "Tabella": tabella, 
             "Totale Records": totale, 
             "Numero Colonne": num_colonne
            }
        )

    # FOGLIO DI RIEPILOGO — fuori dal loop, dopo che stat è completo
    df_stat = pd.DataFrame(stat)
    df_stat.to_excel(writer, sheet_name="Riepilogo", index=False)

C:\Users\pietr\AppData\Local\Temp\ipykernel_16228\2648571197.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


CHIUSURA FILE

In [10]:
conn.close()
print("Connessione chiusa con successo!")

Connessione chiusa con successo!
